# Batch Comparators and the Semantic Judge

## What is a Batch Comparator?

The comparators in previous examples (`exact`, `numeric`, `oneof`) are **per-field
comparators**. Each one scores a single field independently: it takes one gold value,
one extracted value, and returns a score. Fast, local, no side effects.

A **batch comparator** works differently. Instead of scoring fields one at a time, it
receives **all the fields in a record that use it** and scores them together in one call.

Why? Some comparisons need cross-field context. An LLM judge comparing free text
benefits from seeing all the semantic fields in a record at once -- it can make more
consistent judgments, or user pays for only one API call per record instead of one
per field, or for performance reasons, several fields are scored together in a single call is faster.
Good examples using batch comparators include: llm judges for free-text comparison, embeddings, or a combined field like "family_name+given_name" that needs to be scored as a unit.

## The Semantic Comparator

The `semantic` comparator is the built-in batch comparator for free-text comparison.
It wraps an LLM judge that decides whether two texts mean the same thing.

- "Stable film with uniform coverage" vs "Film is stable with even coating" -- **same meaning, score 1.0**
- "Partial success, some defects" vs "Film had major defects throughout" -- **different meaning, score 0.0**
- Exact string matches **short-circuit** -- the LLM is never called when gold == extracted

**Requirements:**
- `pip install -e ".[batch]"` (installs `groq` client)
- A Groq API key (free at https://console.groq.com)

## Data

Three records with free-text `observation` and `outcome` fields. The extracted
versions are paraphrases (should match semantically) or factual disagreements
(should not match).

In [6]:
GOLD = [
    {
        "method": "Chemical Vapor Deposition",
        "observation": "Stable film with uniform coverage.",
        "outcome": "Success, target efficiency achieved.",
    },
    {
        "method": "Sputtering",
        "observation": "Thin but patchy coating observed.",
        "outcome": "Partial success, some defects present.",
    },
    {
        "method": "Pulsed Laser Deposition",
        "observation": "Smooth crystalline film.",          # exact match in extracted
        "outcome": "Fully successful.",                     # exact match in extracted
    },
]

EXTRACTED = [
    {
        "method": "Chemical Vapor Deposition",
        "observation": "Film is stable with even coating.",           # paraphrase -> should match
        "outcome": "Experiment succeeded, reached target efficiency.", # paraphrase -> should match
    },
    {
        "method": "Sputtering",
        "observation": "Coating was thin and had visible patches.",   # paraphrase -> should match
        "outcome": "Film had major defects throughout.",               # factual disagreement -> should NOT match
    },
    {
        "method": "Pulsed Laser Deposition",
        "observation": "Smooth crystalline film.",                    # exact match -> short-circuits LLM
        "outcome": "Fully successful.",                               # exact match -> short-circuits LLM
    },
]

## Step 1: Without Semantic -- exact comparison

First, let's see what happens with the default `exact` comparator on free-text fields.

In [7]:
from struct_extract_eval import infer_schema, annotate_xeval, evaluate

schema_exact = infer_schema(GOLD)
annotate_xeval(schema_exact)

run_exact = evaluate(GOLD, EXTRACTED, schema=schema_exact)

print("With exact comparator (default):")
print(f"  F1: {run_exact.mean_f1:.3f}  P: {run_exact.mean_precision:.3f}  R: {run_exact.mean_recall:.3f}")
print()
for record in run_exact.records:
    print(f"  Record {record.record_id}: F1={record.f1:.3f}")
    for fr in record.field_results:
        if fr.path in ("observation", "outcome"):
            print(f"    {fr.path}: {fr.status} (score={fr.score:.1f})")

With exact comparator (default):
  F1: 0.556  P: 0.556  R: 0.556

  Record 0: F1=0.333
    observation: mismatch (score=0.0)
    outcome: mismatch (score=0.0)
  Record 1: F1=0.333
    observation: mismatch (score=0.0)
    outcome: mismatch (score=0.0)
  Record 2: F1=1.000
    observation: match (score=1.0)
    outcome: match (score=1.0)


Records 0 and 1 have paraphrased text -- semantically correct but scored as mismatches
by `exact`. Only record 2 matches because the text is identical.

## Step 2: Register the Semantic Comparator

The semantic comparator wraps an LLM judge. You choose the judge implementation and
register it under a name (typically `"semantic"`).

This package provides `GroqJudge`, which uses the Groq API with Llama 3.3 70B. Replace the API key below with your own when use it.

In [11]:
from struct_extract_eval.batch import GroqJudge, SemanticBatchComparator
from struct_extract_eval.core.comparators.registry import register

# Replace with your Groq API key (free at https://console.groq.com)
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

judge = GroqJudge(api_key=GROQ_API_KEY)
register("semantic", SemanticBatchComparator(judge), overwrite=True)
print("Registered 'semantic' comparator with GroqJudge.")

Registered 'semantic' comparator with GroqJudge.


## Step 3: Update the Schema

Set `observation` and `outcome` to use the `semantic` comparator.
`method` stays as `exact` -- it's a structured value, not free text.

In [12]:
from copy import deepcopy

schema_semantic = deepcopy(schema_exact)
schema_semantic["properties"]["observation"]["x-eval-compare"] = "semantic"
schema_semantic["properties"]["outcome"]["x-eval-compare"] = "semantic"

import json
print(json.dumps(schema_semantic, indent=2))

{
  "type": "object",
  "properties": {
    "method": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "observation": {
      "type": "string",
      "x-eval-compare": "semantic"
    },
    "outcome": {
      "type": "string",
      "x-eval-compare": "semantic"
    }
  }
}


## Step 4: Evaluate with Semantic

In [13]:
run_semantic = evaluate(GOLD, EXTRACTED, schema=schema_semantic)

print("With semantic comparator:")
print(f"  F1: {run_semantic.mean_f1:.3f}  P: {run_semantic.mean_precision:.3f}  R: {run_semantic.mean_recall:.3f}")
print()
for record in run_semantic.records:
    print(f"  Record {record.record_id}: F1={record.f1:.3f}")
    for fr in record.field_results:
        if fr.path in ("observation", "outcome"):
            print(f"    {fr.path}: {fr.status} (score={fr.score:.1f}, reason={fr.reason})")

With semantic comparator:
  F1: 0.889  P: 0.889  R: 0.889

  Record 0: F1=1.000
    observation: match (score=1.0, reason=judge match)
    outcome: match (score=1.0, reason=judge match)
  Record 1: F1=0.667
    observation: match (score=1.0, reason=judge match)
    outcome: mismatch (score=0.0, reason=judge mismatch)
  Record 2: F1=1.000
    observation: match (score=1.0, reason=exact)
    outcome: match (score=1.0, reason=exact)


## What Happened

**Record 0:** Both `observation` and `outcome` are paraphrases. The LLM judge recognizes
they mean the same thing and scores them as matches.

**Record 1:** `observation` is a paraphrase (match). But `outcome` is a factual
disagreement -- "Partial success, some defects" vs "Film had major defects throughout."
The LLM judge correctly scores this as a mismatch.

**Record 2:** Both fields are exact string matches. The semantic comparator
**short-circuits** -- it sees the strings are identical and returns 1.0 without
calling the LLM at all. This is the highest-leverage optimization.

## How Batching Works

The semantic comparator collects ALL semantic fields in one record and sends them
to the LLM in a **single call**:

```
Record 0: observation + outcome -> 1 LLM call (2 fields)
Record 1: observation + outcome -> 1 LLM call (2 fields)
Record 2: observation + outcome -> 0 LLM calls (both exact match, short-circuited)
```

Total: **2 LLM calls** for 6 semantic fields. Without batching, it would be 4 calls.
Without short-circuiting, it would be 6 calls.

## What's Next?

The semantic comparator treats each field independently -- it judges `observation`
and `outcome` as separate questions. But sometimes fields are **not independent**.

For example, `family_name` and `given_name` should be scored together, only  `family_name`
is correct will refer to another person, user might want to give partial credit, etc.
Or maybe you have a group of related fields that together determine the meaning of a record,
and you want to score them as a unit.

**Compound comparators** (next example: `05_example_compound`) solve this by grouping
sibling fields and scoring them as a unit. They are also batch comparators, but instead
of sending fields to an LLM, they apply custom logic across the group.